In [1]:
%%capture
%pip install deepagents langchain-openai

In [2]:
import os
from dotenv import load_dotenv
from deepagents import create_deep_agent


load_dotenv()


def calcular_imc(peso_kg: float, altura_m: float) -> str:
    """Calcula o Índice de Massa Corporal (IMC) de uma pessoa."""
    imc = peso_kg / (altura_m ** 2)
    if imc < 18.5:
        categoria = "Abaixo do peso"
    elif imc < 25:
        categoria = "Peso normal"
    elif imc < 30:
        categoria = "Sobrepeso"
    else:
        categoria = "Obesidade"
    return f"IMC: {imc:.1f} — {categoria}"


agente = create_deep_agent(
    model="openai:gpt-4o",
    tools=[calcular_imc],
    system_prompt="Você é um assistente de saúde simpático que responde em português.",
)

resultado = agente.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Meu peso é 75 kg e minha altura é 1.72 m. Qual é o meu IMC?"
        }
    ]
})

ultima_mensagem = resultado["messages"][-1]
print(ultima_mensagem.content)

c:\Users\renan\AppData\Local\Programs\Python\Python312\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


[{'type': 'text', 'text': 'Seu IMC é 25.4, o que indica sobrepeso.', 'annotations': [], 'id': 'msg_040f669743d30fe80069fe99d83d108193a889805bb948be82'}]


In [4]:
"""
Exemplo deepagents com streaming (ver o "pensamento" do agente)
===============================================================

Pré-requisitos:
    pip install deepagents langchain-openai python-dotenv

Defina sua chave de API no arquivo .env:
    OPENAI_API_KEY=sk-...
"""

import os
from dotenv import load_dotenv
from deepagents import create_deep_agent

load_dotenv()


def calcular_imc(peso_kg: float, altura_m: float) -> str:
    """Calcula o Índice de Massa Corporal (IMC) de uma pessoa."""
    imc = peso_kg / (altura_m ** 2)
    if imc < 18.5:
        categoria = "Abaixo do peso"
    elif imc < 25:
        categoria = "Peso normal"
    elif imc < 30:
        categoria = "Sobrepeso"
    else:
        categoria = "Obesidade"
    return f"IMC: {imc:.1f} — {categoria}"


# ── Criar o agente ──────────────────────────────────────────────
agente = create_deep_agent(
    model="openai:gpt-4o",
    tools=[calcular_imc],
    system_prompt="Você é um assistente de saúde simpático que responde em português.",
)

mensagens = {
    "messages": [
        {
            "role": "user",
            "content": "Meu peso é 75 kg e minha altura é 1.72 m. Qual é o meu IMC?"
        }
    ]
}

In [5]:
%%capture
%pip install tavily-python

In [6]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from tavily import TavilyClient

load_dotenv()

model = init_chat_model("openai:gpt-4o")
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


# ── Ferramenta compartilhada ────────────────────────────────────
def pesquisar_web(query: str) -> str:
    """Pesquisa informações na web."""
    results = tavily.search(query, max_results=3)
    return "\n".join(r["content"] for r in results["results"])


# ═════════════════════════════════════════════════════════════════
# REACT AGENT (LangGraph básico)
# ═════════════════════════════════════════════════════════════════
# O ReAct é um loop simples: Pensar → Agir → Observar → Repetir
# Ele NÃO tem planejamento, NÃO escreve arquivos, NÃO delega.

from langgraph.prebuilt import create_react_agent

react_agent = create_react_agent(
    model=model,
    tools=[pesquisar_web],
    prompt="Você é um assistente de pesquisa. Responda em português.",
)

# Internamente, o fluxo do ReAct é:
#
#   Usuário ──► LLM decide ──► Chama ferramenta ──► LLM lê resultado ──► Responde
#                   ▲                                      │
#                   └──────────────────────────────────────┘
#                         (repete até ter a resposta)


# ═════════════════════════════════════════════════════════════════
# DEEP AGENT (deepagents)
# ═════════════════════════════════════════════════════════════════
# O Deep Agent adiciona 4 capacidades ao loop ReAct:
#   1. Planejamento (write_todos / read_todos)
#   2. Sistema de arquivos (read_file, write_file, edit_file)
#   3. Sub-agentes (task) para delegar subtarefas
#   4. Sumarização automática quando o contexto fica grande

from deepagents import create_deep_agent

deep_agent = create_deep_agent(
    model=model,
    tools=[pesquisar_web],
    system_prompt="Você é um assistente de pesquisa. Responda em português.",
)

# Internamente, o fluxo do Deep Agent é:
#
#   Usuário ──► LLM cria PLANO (todos) ──► Para cada item do plano:
#                                              │
#                                              ├─ Chama ferramenta
#                                              ├─ Salva resultado em ARQUIVO
#                                              ├─ Pode DELEGAR para sub-agente
#                                              └─ Marca item como concluído
#                                              │
#                                          [Se contexto grande → SUMARIZA]
#                                              │
#                                          ──► Resposta final compilada


# ═════════════════════════════════════════════════════════════════
# TESTE: mesma pergunta para os dois
# ═════════════════════════════════════════════════════════════════

PERGUNTA = (
    "Compare as vantagens e desvantagens de Python vs Rust "
    "para desenvolvimento de APIs. Escreva um resumo estruturado."
)

mensagens = {"messages": [{"role": "user", "content": PERGUNTA}]}


def rodar_com_streaming(agent, nome: str):
    """Roda o agente mostrando cada passo."""
    print(f"\n{'═' * 60}")
    print(f" {nome}")
    print(f"{'═' * 60}")

    for chunk in agent.stream(mensagens, stream_mode="updates"):
        for node_name, state_update in chunk.items():
            if "messages" not in state_update:
                continue
            for msg in state_update["messages"]:
                # Chamada de ferramenta
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"  🔧 {tc['name']}({list(tc['args'].keys())})")

                # Resultado de ferramenta
                elif msg.type == "tool":
                    preview = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
                    print(f"  📋 → {preview}")

                # Resposta do agente
                elif hasattr(msg, "content") and msg.content and msg.type == "ai":
                    print(f"\n  💬 Resposta final:")
                    print(f"  {msg.content[:500]}...")
                    if len(msg.content) > 500:
                        print(f"  [...{len(msg.content)} caracteres no total]")

    print()


# ── Rodar ambos ─────────────────────────────────────────────────
if __name__ == "__main__":
    rodar_com_streaming(react_agent, "REACT AGENT (simples)")
    rodar_com_streaming(deep_agent, "DEEP AGENT (com planejamento)")

C:\Users\renan\AppData\Local\Temp\ipykernel_9928\2731277375.py:27: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  react_agent = create_react_agent(



════════════════════════════════════════════════════════════
 REACT AGENT (simples)
════════════════════════════════════════════════════════════
  🔧 pesquisar_web(['query'])
  🔧 pesquisar_web(['query'])
  🔧 pesquisar_web(['query'])
  🔧 pesquisar_web(['query'])
  📋 → Alta performance e baixa latência: o rust é excelente para lidar com aplicações ...
  📋 → Desvantagem 1: desempenho e eficiência. O desempenho é um dos principais pontos ...
  📋 → O rust possui muitas garantias de segurança que são verificadas em tempo de comp...
  📋 → Python is a top pick for building APIs because it is super easy to read and writ...

  💬 Resposta final:
  ### Python para Desenvolvimento de APIs

#### Vantagens:
1. **Facilidade de Uso e Leitura**: Python é conhecido por sua sintaxe clara e fácil de entender, o que facilita a escrita e leitura de código.
2. **Grande Comunidade e Suporte**: Existe um vasto ecossistema de bibliotecas e uma comunidade ativa que pode ajudar no desenvolvimento de APIs.
3. **Fer

TypeError: argument of type 'NoneType' is not iterable